In [9]:
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import scanpy as sc

In [7]:
segmentation_path = Path("/mnt/c/Users/jonan/Documents/1Work/Roselab/Spatial/CAR_T/data/cell_segmentation/")
adata_file        = segmentation_path / "concatenated" / "combined_adata.h5ad"
outdir            = Path("/mnt/c/Users/jonan/Documents/1Work/RoseLab/Spatial/CAR_T/Results/QC_figures/")

# Load
ST_sample = sc.read_h5ad(adata_file)

# Quick sanity
print(ST_sample)
print("obs columns:", list(ST_sample.obs.columns)[:20], "...")
print("var shape:", ST_sample.var.shape)

AnnData object with n_obs × n_vars = 200385 × 19059
    obs: 'id', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mouse', 'sample_id', 'condition', 'cx', 'cy', 'TMA'
obs columns: ['id', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mouse', 'sample_id', 'condition', 'cx', 'cy', 'TMA'] ...
var shape: (19059, 0)


In [10]:
# Use CSR for speed
X = ST_sample.X
if sp.issparse(X):
    X = X.tocsr()
else:
    X = sp.csr_matrix(X)

# total_counts
if "total_counts" not in ST_sample.obs.columns:
    ST_sample.obs["total_counts"] = np.asarray(X.sum(axis=1)).ravel()

# n_genes_by_counts
if "n_genes_by_counts" not in ST_sample.obs.columns:
    if sp.issparse(X):
        ST_sample.obs["n_genes_by_counts"] = np.diff(X.indptr)
    else:
        ST_sample.obs["n_genes_by_counts"] = (X > 0).sum(axis=1)

# Mito/ribo detection
# Priority 1: var flags if present; else fallback to name prefixes.
var_names = ST_sample.var_names.astype(str)

if "mt" in ST_sample.var.columns and ST_sample.var["mt"].dtype == bool:
    mito_mask = ST_sample.var["mt"].to_numpy()
else:
    mito_prefixes = ("mt-", "MT-", "Mt-", "mt_", "MT_")
    mito_mask = np.zeros(ST_sample.n_vars, dtype=bool)
    for p in mito_prefixes:
        mito_mask |= np.char.startswith(var_names.to_numpy(), p)

if "ribo" in ST_sample.var.columns and ST_sample.var["ribo"].dtype == bool:
    ribo_mask = ST_sample.var["ribo"].to_numpy()
else:
    # Mouse-style defaults (Rpl/Rps). Adjust if using human (RPL/RPS already covered).
    ribo_prefixes = ("Rpl", "Rps", "RPL", "RPS")
    ribo_mask = np.zeros(ST_sample.n_vars, dtype=bool)
    for p in ribo_prefixes:
        ribo_mask |= np.char.startswith(var_names.to_numpy(), p)

# Fractions
total = ST_sample.obs["total_counts"].to_numpy().astype(float)
mito_sum = np.asarray(X[:, mito_mask].sum(axis=1)).ravel() if mito_mask.any() else np.zeros(ST_sample.n_obs)
ribo_sum = np.asarray(X[:, ribo_mask].sum(axis=1)).ravel() if ribo_mask.any() else np.zeros(ST_sample.n_obs)

with np.errstate(divide="ignore", invalid="ignore"):
    ST_sample.obs["pct_mt"]   = np.where(total > 0, 100.0 * mito_sum / total, 0.0)
    ST_sample.obs["pct_ribo"] = np.where(total > 0, 100.0 * ribo_sum / total, 0.0)

ST_sample.obs[["total_counts","n_genes_by_counts","pct_mt","pct_ribo"]].head()


TypeError: string operation on non-string array